In [11]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import os

from src.config import Configuration

CONFIG = Configuration()

# Visualize filters

In [13]:
from src.model import load_cascade, CascadeClassifier

cascade_path = os.path.join(CONFIG.computed_haar_cascades, 'haar_cascade_stage_13_fpr_0.0049.xml')
# cascade_path = os.path.join(CONFIG.cv_haar_cascades, 'haarcascade_frontalface_default.xml')

cascade = load_cascade(cascade_path)
CONFIG.crop_size = max(cascade.height, cascade.width)
classifier = CascadeClassifier(CONFIG, cascade)

Loading Haar cascade from: ../models/haar_cascades_computed/haar_cascade_stage_13_fpr_0.0049.xml


In [ ]:
import math

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import ipywidgets as widgets
from IPython.display import display


def feature_to_filter_map(feature, width, height):
    filter_map = np.zeros((height, width), dtype=np.float32)

    for rect in feature.rectangles:
        x0 = int(rect.x)
        y0 = int(rect.y)
        x1 = x0 + int(rect.width)
        y1 = y0 + int(rect.height)
        filter_map[y0:y1, x0:x1] += float(rect.weight)

    return filter_map


def extract_stage_filters(cascade):
    stages = []

    for stage in cascade.stages:
        filters = []

        for classifier in stage.weak_classifiers:
            if classifier.feature is None:
                continue

            filters.append(
                {
                    "classifier_id": classifier.classifier_id,
                    "feature_id": classifier.feature_id,
                    "threshold": float(classifier.threshold),
                    "left_value": float(classifier.left_value),
                    "right_value": float(classifier.right_value),
                    "rectangles": [
                        {
                            "x": int(rect.x),
                            "y": int(rect.y),
                            "width": int(rect.width),
                            "height": int(rect.height),
                            "weight": float(rect.weight),
                        }
                        for rect in classifier.feature.rectangles
                    ],
                    "filter_map": feature_to_filter_map(
                        classifier.feature,
                        width=cascade.width,
                        height=cascade.height,
                    ),
                }
            )

        stages.append(
            {
                "stage_id": stage.stage_id,
                "threshold": float(stage.threshold),
                "max_weak_count": stage.max_weak_count,
                "filters": filters,
            }
        )

    return {
        "cascade_type": cascade.cascade_type,
        "feature_type": cascade.feature_type,
        "width": cascade.width,
        "height": cascade.height,
        "stage_count": len(stages),
        "stages": stages,
    }


stage_filters = extract_stage_filters(cascade)


def plot_stage(stage_index):
    stage = stage_filters["stages"][stage_index]
    filters = stage["filters"]

    plt.close("all")
    plt.style.use("dark_background")

    if not filters:
        print(f"Stage {stage['stage_id']} has no filters.")
        return

    n_filters = len(filters)
    n_cols = min(4, max(1, int(math.ceil(math.sqrt(n_filters)))))
    n_rows = int(math.ceil(n_filters / n_cols))
    vmax = max(1e-9, max(float(np.max(np.abs(item["filter_map"]))) for item in filters))
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(3.2 * n_cols, 3.2 * n_rows),
        squeeze=False,
    )
    fig.patch.set_facecolor("#111827")
    fig.subplots_adjust(left=0.04, right=0.88, top=0.90, bottom=0.06, wspace=0.22, hspace=0.28)
    fig.suptitle(
        f"Stage {stage['stage_id']} | threshold={stage['threshold']:.6f} | filters={n_filters}",
        fontsize=14,
        color="white",
    )

    for ax, item in zip(axes.ravel(), filters):
        image = ax.imshow(
            item["filter_map"],
            cmap="RdBu_r",
            norm=norm,
            interpolation="nearest",
        )
        ax.set_facecolor("#0f172a")
        ax.set_title(f"clf {item['classifier_id']} | feat {item['feature_id']}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_visible(False)
        ax.spines["left"].set_visible(False)
        ax.title.set_color("white")

    cax = fig.add_axes([0.91, 0.12, 0.02, 0.76])
    fig.colorbar(image, cax=cax)
    cax.set_facecolor("#111827")

    for ax in axes.ravel()[n_filters:]:
        ax.axis("off")

    plt.show()


stage_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=max(0, stage_filters["stage_count"] - 1),
    step=1,
    description="Stage",
    continuous_update=False,
)

stage_output = widgets.interactive_output(plot_stage, {"stage_index": stage_slider})
display(widgets.VBox([stage_slider, stage_output]))


## How to read the colors

These filter images use a diverging colormap centered at **0**.

- **Blue tones**: negative rectangle contribution (subtracts from the feature response).
- **Red tones**: positive rectangle contribution (adds to the feature response).
- **Near white / very light**: value close to zero (little or no contribution).
- **Darker, more saturated color**: larger absolute contribution (stronger effect).

So intensity means **strength**, and color hue (red vs blue) means **direction** (add vs subtract).